# 🛒 E-commerce Customer Churn Prediction

**Dataset**: [Ecommerce Customer Behavior Dataset](https://www.kaggle.com/datasets/dhairyajeetsingh/ecommerce-customer-behavior-dataset)

- **50,000** customers | **25** features | Binary target: `Churned` (0=Active, 1=Churned)
- Features: Demographics, Platform Engagement, Purchase Behavior, Customer Service

**Approach**: Train multiple ML models (Logistic Regression, Random Forest, XGBoost) and compare accuracy.

## Step 1: Setup & Install Dependencies

In [ ]:
!pip install -q kaggle xgboost

## Step 2: Download Dataset from Kaggle

Upload your `kaggle.json` API key file. Get it from: [kaggle.com/settings](https://www.kaggle.com/settings) → **Create New Token**

In [ ]:
import os
from google.colab import files

uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle API key configured!')

In [ ]:
!kaggle datasets download -d dhairyajeetsingh/ecommerce-customer-behavior-dataset --unzip -p /content/dataset
print('\n✅ Dataset downloaded and extracted!')

## Step 3: Load & Explore the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
csv_files = [f for f in os.listdir('/content/dataset') if f.endswith('.csv')]
print(f'📂 Files found: {csv_files}')

df = pd.read_csv(f'/content/dataset/{csv_files[0]}')

print(f'\n📊 Dataset Shape: {df.shape}')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.shape[1]}')
print(f'\n📋 Columns: {list(df.columns)}')
df.head()

In [ ]:
# Dataset info
print('📊 Dataset Info:')
print('=' * 60)
print(df.info())
print('\n📊 Statistical Summary:')
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print('⚠️ Missing Values:')
    print(missing_df)
else:
    print('✅ No missing values found!')

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
churn_counts = df['Churned'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Active (0)', 'Churned (1)'], churn_counts.values, color=colors, edgecolor='black')
for i, (count, pct) in enumerate(zip(churn_counts.values, churn_counts.values / len(df) * 100)):
    axes[0].text(i, count + 200, f'{count:,}\n({pct:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('🎯 Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Active', 'Churned'], autopct='%1.1f%%',
            colors=colors, explode=[0, 0.05], shadow=True, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('🎯 Churn Ratio', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical features distribution
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols = [c for c in numerical_cols if c != 'Churned']

n_cols = 4
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    df[col].hist(bins=30, ax=axes[i], color='#3498db', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Count')

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📊 Numerical Features Distribution', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Churn analysis by categorical features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'📋 Categorical columns: {categorical_cols}')

if len(categorical_cols) > 0:
    fig, axes = plt.subplots(1, min(len(categorical_cols), 3), figsize=(18, 5))
    if len(categorical_cols) == 1:
        axes = [axes]

    for i, col in enumerate(categorical_cols[:3]):
        churn_by_cat = df.groupby(col)['Churned'].mean().sort_values(ascending=False).head(10)
        churn_by_cat.plot(kind='bar', ax=axes[i], color='#e74c3c', edgecolor='black', alpha=0.8)
        axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
        axes[i].set_ylabel('Churn Rate')
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(16, 12))
corr_cols = numerical_cols + ['Churned']
corr_matrix = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('🔗 Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
print('\n🎯 Top Features Correlated with Churn:')
churn_corr = corr_matrix['Churned'].drop('Churned').abs().sort_values(ascending=False)
for feat, corr_val in churn_corr.head(10).items():
    direction = '↑' if corr_matrix.loc[feat, 'Churned'] > 0 else '↓'
    print(f'   {direction} {feat}: {corr_matrix.loc[feat, "Churned"]:.4f}')

## Step 5: Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Create a working copy
data = df.copy()

# Handle missing values
for col in data.columns:
    if data[col].isnull().sum() > 0:
        if data[col].dtype in ['float64', 'int64']:
            data[col].fillna(data[col].median(), inplace=True)
        else:
            data[col].fillna(data[col].mode()[0], inplace=True)

print(f'✅ Missing values handled. Remaining: {data.isnull().sum().sum()}')

# Encode categorical variables
label_encoders = {}
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le
    print(f'   Encoded {col}: {len(le.classes_)} categories')

# Separate features and target
X = data.drop('Churned', axis=1)
y = data['Churned']

print(f'\n📊 Features shape: {X.shape}')
print(f'   Target distribution: {dict(y.value_counts())}')

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\n📊 Data Split:')
print(f'   Training: {X_train.shape[0]:,} samples')
print(f'   Testing:  {X_test.shape[0]:,} samples')

## Step 6: Train Multiple Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import time

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False,
                              eval_metric='logloss', verbosity=0),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
}

# Train and evaluate all models
results = {}
print('🚀 Training Models...\n')
print(f'{"Model":<25} {"Accuracy":>10} {"AUC-ROC":>10} {"Time (s)":>10}')
print('=' * 58)

for name, model in models.items():
    start = time.time()

    # Use scaled data for Logistic Regression, KNN, SVM; raw for tree-based
    if name in ['Logistic Regression', 'KNN']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    elapsed = time.time() - start
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        'model': model,
        'accuracy': acc,
        'auc_roc': auc,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'time': elapsed
    }

    print(f'{name:<25} {acc:>9.4f} {auc:>10.4f} {elapsed:>9.2f}s')

# Best model
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_acc = results[best_model_name]['accuracy']
print(f'\n🏆 Best Model: {best_model_name} (Accuracy: {best_acc*100:.2f}%)')

## Step 7: Model Comparison Visualization

In [ ]:
# Accuracy comparison bar chart
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] * 100 for m in model_names]
auc_scores = [results[m]['auc_roc'] * 100 for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
colors = ['#2ecc71' if m == best_model_name else '#3498db' for m in model_names]
bars = axes[0].barh(model_names, accuracies, color=colors, edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2.,
                 f'{acc:.2f}%', va='center', fontweight='bold')
axes[0].set_xlabel('Accuracy (%)', fontsize=12)
axes[0].set_title('🎯 Model Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, max(accuracies) + 5)

# AUC-ROC
bars2 = axes[1].barh(model_names, auc_scores, color=colors, edgecolor='black', linewidth=0.5)
for bar, auc in zip(bars2, auc_scores):
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2.,
                 f'{auc:.2f}%', va='center', fontweight='bold')
axes[1].set_xlabel('AUC-ROC (%)', fontsize=12)
axes[1].set_title('📈 AUC-ROC Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, max(auc_scores) + 5)

plt.tight_layout()
plt.show()

## Step 8: Best Model - Detailed Evaluation

In [ ]:
# Best model results
best = results[best_model_name]
y_pred_best = best['y_pred']

print(f'🏆 Best Model: {best_model_name}')
print('=' * 50)
print(f'   Accuracy: {best["accuracy"]*100:.2f}%')
print(f'   AUC-ROC:  {best["auc_roc"]*100:.2f}%')

print('\n📋 Classification Report:\n')
print(classification_report(y_test, y_pred_best, target_names=['Active', 'Churned']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active', 'Churned'], yticklabels=['Active', 'Churned'],
            linewidths=0.5, linecolor='gray')
axes[0].set_title(f'🔍 Confusion Matrix - {best_model_name}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='YlOrRd', ax=axes[1],
            xticklabels=['Active', 'Churned'], yticklabels=['Active', 'Churned'],
            linewidths=0.5, linecolor='gray')
axes[1].set_title(f'🔍 Normalized Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Step 9: ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(10, 7))
colors_roc = plt.cm.Set1(np.linspace(0, 1, len(results)))

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    lw = 3 if name == best_model_name else 1.5
    style = '-' if name == best_model_name else '--'
    plt.plot(fpr, tpr, color=color, lw=lw, linestyle=style,
             label=f'{name} (AUC = {res["auc_roc"]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('📈 ROC Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 10: Feature Importance

In [ ]:
# Feature importance from best tree-based model
tree_models = ['Random Forest', 'XGBoost', 'Gradient Boosting', 'Decision Tree']
fi_model_name = best_model_name if best_model_name in tree_models else 'Random Forest'
fi_model = results[fi_model_name]['model']

importances = fi_model.feature_importances_
feature_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
colors_fi = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_imp)))
plt.barh(feature_imp['Feature'], feature_imp['Importance'], color=colors_fi, edgecolor='black', linewidth=0.5)
plt.xlabel('Importance', fontsize=12)
plt.title(f'🔑 Feature Importance ({fi_model_name})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n🔝 Top 5 Most Important Features:')
for _, row in feature_imp.tail(5).iloc[::-1].iterrows():
    print(f'   {row["Feature"]}: {row["Importance"]:.4f}')

## Step 11: Summary

In [ ]:
print('\n' + '=' * 60)
print('📋 FINAL SUMMARY')
print('=' * 60)
print(f'  Dataset:         Ecommerce Customer Behavior Dataset')
print(f'  Total Samples:   {len(df):,}')
print(f'  Features:        {X.shape[1]}')
print(f'  Target:          Churned (Binary: Active/Churned)')
print(f'  Train Samples:   {len(X_train):,}')
print(f'  Test Samples:    {len(X_test):,}')
print(f'\n  🏆 Best Model:   {best_model_name}')
print(f'  Best Accuracy:   {results[best_model_name]["accuracy"]*100:.2f}%')
print(f'  Best AUC-ROC:    {results[best_model_name]["auc_roc"]*100:.2f}%')
print(f'\n  📊 All Model Results:')
for name, res in sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True):
    marker = '🥇' if name == best_model_name else '  '
    print(f'  {marker} {name:<25} Acc: {res["accuracy"]*100:.2f}%  AUC: {res["auc_roc"]*100:.2f}%')
print('=' * 60)

## Step 12: Save the Best Model

In [ ]:
import joblib
import json

os.makedirs('/content/saved_model', exist_ok=True)

# Save best model
model_path = f'/content/saved_model/best_model_{best_model_name.lower().replace(" ", "_")}.pkl'
joblib.dump(results[best_model_name]['model'], model_path)
print(f'✅ Best model saved: {model_path}')

# Save scaler
scaler_path = '/content/saved_model/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'✅ Scaler saved: {scaler_path}')

# Save label encoders
encoders_path = '/content/saved_model/label_encoders.pkl'
joblib.dump(label_encoders, encoders_path)
print(f'✅ Label encoders saved: {encoders_path}')

# Save model info
model_info = {
    'best_model': best_model_name,
    'accuracy': results[best_model_name]['accuracy'],
    'auc_roc': results[best_model_name]['auc_roc'],
    'features': list(X.columns),
    'categorical_columns': categorical_cols,
    'all_results': {name: {'accuracy': res['accuracy'], 'auc_roc': res['auc_roc']}
                    for name, res in results.items()}
}
info_path = '/content/saved_model/model_info.json'
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
print(f'✅ Model info saved: {info_path}')

# Zip and download
!cd /content && zip -r saved_model.zip saved_model/

from google.colab import files
files.download('/content/saved_model.zip')
print('\n✅ Model downloaded!')